In [1]:
import pandas as pd

df = pd.read_csv('../data/clean/superstore_cleaned.csv')
print(df.shape)
df.head()

(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


q1: Total sales, profit and margin by year

In [ ]:
df['Order Date']=pd.to_datetime(df['Order Date'])
df['year']=df['Order Date'].dt.year

q1=df.groupby('year').agg(
    total_sales=('Sales','sum'),
    total_profit=('Profit','sum')
).reset_index()

q1['profit_margin_percent']=(
    q1['total_profit']/q1['total_sales']*100
).round(2)

q1

,year,total_sales,total_profit,profit_margin_percent
0,2014,484247.4981,49543.9741,10.23
1,2015,470532.5090,61618.6037,13.10
2,2016,609205.5980,81795.1743,13.43
3,2017,733215.2552,93439.2696,12.74


Q2: Revenue per month across all four years

In [3]:
df['month']=df['Order Date'].dt.to_period('M')
q2=df.groupby('month').agg(
    month_revenue=('Sales','sum')
).reset_index()

q2['month_revenue']=q2['month_revenue'].round(2)

q2

,month,month_revenue
0,2014-01,14236.90
1,2014-02,4519.89
2,2014-03,55691.01
3,2014-04,28295.34
4,2014-05,23648.29
5,2014-06,34595.13
6,2014-07,33946.39
7,2014-08,27909.47
8,2014-09,81777.35
9,2014-10,31453.39


q3: Top 5 customers by total spend

In [8]:
q3=df.groupby('Customer Name').agg(
    total_spend=('Sales','sum')
).reset_index()
q3=q3.sort_values('total_spend',ascending=False).head(5).reset_index(drop=True)

q3['total_spend']=q3['total_spend'].round(2)

q3

,Customer Name,total_spend
0,Sean Miller,25043.05
1,Tamara Chand,19052.22
2,Raymond Buch,15117.34
3,Tom Ashbrook,14595.62
4,Adrian Barton,14473.57


--q4:Profit by region and category

In [10]:
q4=df.groupby(['Region','Category']).agg(
    total_profit=('Profit','sum')
).reset_index()
q4['total_profit']=q4['total_profit'].round(2)

q4

,Region,Category,total_profit
0,Central,Furniture,-2871.05
1,Central,Office Supplies,8879.98
2,Central,Technology,33697.43
3,East,Furniture,3046.17
4,East,Office Supplies,41014.58
5,East,Technology,47462.04
6,South,Furniture,6771.21
7,South,Office Supplies,19986.39
8,South,Technology,19991.83
9,West,Furniture,11504.95


--q5:Sub-categories with negative total profit

In [15]:
q5=df.groupby('Sub-Category').agg(
    total_profit=('Profit','sum')
).reset_index()
q5=q5[q5['total_profit']<0]
q5=q5.sort_values('total_profit',ascending=False).reset_index(drop=True)
q5['total_profit']=q5['total_profit'].round(2)

q5

,Sub-Category,total_profit
0,Supplies,-1189.10
1,Bookcases,-3472.56
2,Tables,-17725.48



--q6:Average discount on loss-making vs profitable orders

In [18]:
df['order_type']=df['Profit'].apply(
    lambda x:'loss-making' if x<0 else 'profitable'
)
q6=df.groupby('order_type').agg(
    avg_discount=('Discount','mean')
).reset_index()

q6['avg_discount']=q6['avg_discount'].round(3)

q6

,order_type,avg_discount
0,loss-making,0.481
1,profitable,0.081


--q7:Average days from order to ship by ship mode

In [21]:
df['Order Date']=pd.to_datetime(df['Order Date'])
df['Ship Date']=pd.to_datetime(df['Ship Date'])

In [22]:
df['shipping_days']=(df['Ship Date']-df['Order Date']).dt.days
q7=df.groupby('Ship Mode').agg(
    avg_shippingdays=('shipping_days','mean')
).reset_index()
q7['avg_shippingdays']=q7['avg_shippingdays'].round(2)

q7=q7.sort_values('avg_shippingdays').reset_index(drop=True)

q7

,Ship Mode,avg_shippingdays
0,Same Day,0.04
1,First Class,2.18
2,Second Class,3.24
3,Standard Class,5.01


--q8:Each customer's first order date and number of orders

In [24]:
q8=df.groupby('Customer Name').agg(
    first_order=('Order Date','min'),
    number_of_orders=('Order ID','nunique')
).reset_index()
q8=q8.sort_values('first_order').reset_index(drop=True)

q8

,Customer Name,first_order,number_of_orders
0,Darren Powers,2014-01-03,9
1,Phillina Ober,2014-01-04,5
2,Mick Brown,2014-01-05,7
3,Maria Etezadi,2014-01-06,10
4,Jack O'Briant,2014-01-06,9
...,...,...,...
788,Jenna Caffey,2017-07-08,1
789,Theresa Coyne,2017-09-15,1
790,Patricia Hirasaki,2017-10-21,1
791,Jocasta Rupert,2017-10-23,1
